In [1]:
import json
import os
import re 
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback
import evaluate
from seqeval.metrics import classification_report
import seaborn as sns



INPUT_FILE_PATH = "./new_dataset.json" 
OUTPUT_FILE_PATH = "./training_data_bio_new.json" 



c:\Users\alexi\Desktop\rl_reccomender_course\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
with open(INPUT_FILE_PATH, 'r') as f:
    data = json.load(f) 

print(type(data))
train_dataset = []

<class 'list'>


In [25]:
import string

def norm(token: str) -> str:
    return token.strip(string.punctuation).lower()

def find_span(tokens, surface):
    surf = [norm(t) for t in surface.split()]
    tok_norm = [norm(t) for t in tokens]
    n = len(surf)

    for i in range(len(tok_norm) - n + 1):
        if tok_norm[i:i+n] == surf:
            return i, i + n

    print(tok_norm)
    print(surf)

    return None, None

In [30]:
for sample in data:
    text = sample['text']
    #tokens = sample['tokens']
    tokens = text.split()
    spans = sample['spans']
    to_add = []

    labels = ['O' for _ in range(len(tokens))]

    #print(f"These are lenghts: {len(tokens)}, {len(labels)}")
    if spans != []:
        for skill in spans:
            surface = skill['surface']

            start_token, end_token = find_span(tokens, surface)
            polarity = skill['polarity']

            for i in range(start_token, end_token):
                if i == start_token:
                    labels[i] = 'B-'+polarity+'_SKILL'
                else:
                    labels[i] = 'I-'+polarity+'_SKILL'
    
    to_add.append(tokens)
    to_add.append(labels)

    train_dataset.append(to_add)

with open(OUTPUT_FILE_PATH, 'w', encoding="utf-8") as f:
    json.dump(train_dataset, f, ensure_ascii=False, indent=2)


In [4]:
df = pd.read_csv(filepath_or_buffer="../../Data - Collection/Final/taxonomy.csv")

print(list(df['name']))

['Haskell', 'develop energy saving concepts', 'conduct research on flora', 'Incremental development', 'develop terminology databases', 'KDevelop', 'assess railway operations', 'perform a feasibility study for building management systems', 'Absorb (learning management systems)', 'analyse the artistic concept based on stage actions', 'migrate existing data', 'define database physical structure', 'Maltego', 'search historical sources in archives', 'collect property financial information', 'provide context to news stories', 'circuit diagrams', 'sand blasting machine parts', 'assess visual impact of displays', 'check information on prescriptions', 'Erlang', 'taste wines', 'manage ICT data classification', 'manage claim files', 'information structure', 'communicate gambling rules', 'manage temporary ICT networks for live performance', 'check quality of enamel', 'develop policies for nutritional programs', 'SAS language', 'use an application-specific interface', 'audit contractors', 'define f

In [30]:
def convert_to_bio(task_data: Dict) -> List[Tuple[List[str], List[str]]]:
    """
    Converte una singola task di Label Studio nel formato token/tag BIO.
    
    task_data: Un dizionario che rappresenta una singola riga dal JSON esportato.
    Ritorna: Una lista di coppie (tokens, tags) pronte per l'addestramento.
    """
    all_tasks_bio = []
    
    text = task_data['data']['text']
    annotations = task_data.get('annotations', [{}])[0].get('result', [])
    
    tokens: List[str] = text.split()
    token_tags: List[str] = ['O'] * len(tokens)
    
    current_char_pos = 0
    token_spans = [] 
    for token in tokens:
        match = re.search(re.escape(token), text[current_char_pos:])
        if match:
            start = current_char_pos + match.start()
            end = start + len(token)
            token_spans.append((start, end))
            current_char_pos = end
        else:
            token_spans.append((-1, -1))

    for region in annotations:
        if region['type'] == 'labels':
            start_char = region['value']['start']
            end_char = region['value']['end']
            label = region['value']['labels'][0]
            
            for i, (token_start, token_end) in enumerate(token_spans):
                
                if token_end > start_char and token_start < end_char:
                    
                    # (B)?
                    if token_start >= start_char:
                        if i == 0 or token_tags[i-1] == 'O':
                             token_tags[i] = f'B-{label}'
                        else: # (I)
                             token_tags[i] = f'I-{label}'
                             
                    elif token_tags[i] == 'O':
                        token_tags[i] = f'I-{label}'

    if tokens:
        all_tasks_bio.append((tokens, token_tags))
        
    return all_tasks_bio

In [31]:
bio_data: List[Tuple[List[str], List[str]]] = []

try:
    with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print(f"ERRORE: File non trovato al percorso: {INPUT_FILE_PATH}")
    data = []

for task in data:
    results = convert_to_bio(task)
    bio_data.extend(results)

if bio_data:
    print(f"Conversione completata. Numero totale di frasi processate: {len(bio_data)}")
    print("\nEsempio di output BIO (Token | Tag):")
    for token, tag in zip(bio_data[0][0], bio_data[0][1]):
        print(f"{token.ljust(15)} | {tag}")
else:
    print("path error")

with open(OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
    json.dump(bio_data, f, indent=2)
    
print(f"\nSaved in: {OUTPUT_FILE_PATH}")

Conversione completata. Numero totale di frasi processate: 556

Esempio di output BIO (Token | Tag):
I’m             | O
thinking        | O
about           | O
diving          | O
deeper          | O
into            | O
Kotlin,         | B-INCLUDE_SKILL
but             | O
honestly        | O
I’d             | O
like            | O
to              | O
avoid           | O
C++             | B-AVOID_SKILL
for             | O
now             | O
because         | O
it              | O
stresses        | O
me              | O
out.            | O

Saved in: ./training_data_bio_multi.json


In [16]:
# --- Configuration ---
MODEL_CHECKPOINT = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# The path where you saved your BIO data from Cell 3
INPUT_FILE_PATH = "training_data_bio_multi.json" 
#INPUT_FILE_PATH = "training_data_bio_new.json"
TEST_SET_SIZE = 0.1 # 10% of data for validation/test set

# --- Load and Prepare Data ---
with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
    # The JSON structure is [ [tokens], [tags] ]
    raw_data = json.load(f)

# Convert the list of (tokens, tags) tuples into a Hugging Face Dataset format
# We assume the conversion script outputted a list of lists: [[tokens_1, tags_1], [tokens_2, tags_2], ...]
data_dict = {
    'tokens': [item[0] for item in raw_data],
    'ner_tags_str': [item[1] for item in raw_data] # Temporarily store tags as strings
}

# Create a Dataset object
raw_dataset = Dataset.from_dict(data_dict)

# Split data into training and validation sets
train_test_split = raw_dataset.train_test_split(test_size=TEST_SET_SIZE)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

train_val_split = train_dataset.train_test_split(test_size=0.1)
val_dataset = train_val_split['test']
train_dataset = train_val_split['train']

print(f"Total samples loaded: {len(raw_dataset)}")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}, test samples: {len(test_dataset)}")

Total samples loaded: 556
Training samples: 450, Validation samples: 50, test samples: 56


## Tag Mapping and Tokenization


In [ ]:
# 1. Define the actual labels used in your BIO format
# O (Outside), B-SKILL (Begin-SKILL), I-SKILL (Inside-SKILL)
# Get all unique tags from your loaded data
all_tags = set(tag for sample in raw_dataset['ner_tags_str'] for tag in sample)
label_list = sorted(list(all_tags))
label_map = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label_map.items()} # For output mapping

print("Label Mapping:")
print(label_map)

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# 3. Define Alignment Function
def tokenize_and_align_labels(examples):
    # Tokenize the words (not the raw text, but the already split words)
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label_str_list in enumerate(examples["ner_tags_str"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens (like [CLS], [SEP]) get -100 tag
            if word_idx is None:
                label_ids.append(-100) 
            # Sub-word tokens (part of a word) should keep the tag of the main word
            elif word_idx != previous_word_idx:
                # Assign the original tag's ID (converted from string to int)
                label_ids.append(label_map[label_str_list[word_idx]])
            else:
                # If it's the second or subsequent sub-word piece of the same word, 
                # we keep the previous tag, but change 'B-' to 'I-' if applicable.
                # A simpler approach is to set sub-word tags to -100 to ignore them in loss calculation.
                # Here, we keep the tag ID for demonstration, but if it's the continuation of a word, 
                # it's usually set to -100 or the same tag as the first piece.
                label_ids.append(-100) # Setting to -100 is standard practice for sub-word pieces after the first.
                
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 4. Apply Tokenization and Alignment to the entire dataset
tokenized_train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=True)

print("Tokenization and alignment complete.")

Label Mapping:
{'B-ACQUIRED_SKILL': 0, 'B-AVOID_SKILL': 1, 'B-INCLUDE_SKILL': 2, 'B-NEUTRAL_SKILL': 3, 'I-ACQUIRED_SKILL': 4, 'I-AVOID_SKILL': 5, 'I-INCLUDE_SKILL': 6, 'I-NEUTRAL_SKILL': 7, 'O': 8}


Map: 100%|██████████| 56/56 [00:00<00:00, 2981.93 examples/s]

Tokenization and alignment complete.


##  Model and Metrics Setup
- Load model
- Define evaluation metrics

In [ ]:
# AutoModelForTokenClassification adds a classification head on top of MiniLM
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, 
    num_labels=len(label_list), # Set the output size to the number of unique tags
    id2label=id2label,
    label2id=label_map
)

# We use the seqeval metric standard for NER tasks
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert IDs back to string labels
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Calculate metrics
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Some weights of BertForTokenClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Training Configuration and Execution


In [ ]:

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir="esco_skill_ner_multi_model",               # Output directory for model checkpoints
    learning_rate=2e-4,                              # Standard small learning rate for fine-tuning
    num_train_epochs=40,                              # Number of training epochs 
    per_device_train_batch_size=16,                  # Batch size per GPU/CPU
    per_device_eval_batch_size=16,                   # Evaluation batch size
    weight_decay=0.01,                               # Simple regularization
    eval_strategy="epoch",                     # Evaluate at the end of each epoch
    save_strategy="epoch",                           # Save checkpoint at the end of each epoch
    logging_strategy="epoch",
    load_best_model_at_end=True,                     # Load the model with the best validation performance
    metric_for_best_model="f1",
    push_to_hub=False,                               # Set to True if you want to upload the model to Hugging Face Hub
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] 
)

# 3. Start Training!
print("Starting fine-tuning...")
trainer.train()

print("Fine-tuning complete. Best model saved.")

C:\Users\alexi\AppData\Local\Temp\ipykernel_18520\44554239.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.001400,0.807296,0.169811,0.219512,0.191489,0.738863
2,0.673500,0.688488,0.223301,0.280488,0.248649,0.768049
3,0.512200,0.621749,0.245283,0.317073,0.276596,0.788018
4,0.352400,0.517116,0.333333,0.451220,0.383420,0.844854
5,0.212600,0.514920,0.420000,0.512195,0.461538,0.850998
6,0.109700,0.516301,0.443299,0.524390,0.480447,0.870968
7,0.083900,0.578317,0.454545,0.548780,0.497238,0.866359
8,0.060500,0.538405,0.388350,0.487805,0.432432,0.875576
9,0.055700,0.561494,0.450980,0.560976,0.500000,0.870968
10,0.047900,0.565146,0.386792,0.500000,0.436170,0.878648


c:\Users\alexi\Desktop\rl_reccomender_course\.venv312\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\alexi\Desktop\rl_reccomender_course\.venv312\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\alexi\Desktop\rl_reccomender_course\.venv312\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Fine-tuning complete. Best model saved.


In [6]:
history = trainer.state.log_history

epochs = sorted({h["epoch"] for h in history if "epoch" in h})

rows = []
for e in epochs:
    # tutte le righe di training di quell’epoch
    train_losses = [h["loss"] for h in history 
                    if h.get("epoch") == e and "loss" in h]
    train_loss = float(np.mean(train_losses)) if train_losses else None

    # una riga di eval per epoch (ce n’è una per "eval_loss")
    eval_entry = next(
        (h for h in history 
         if h.get("epoch") == e and "eval_loss" in h),
        {}
    )

    row = {
        "Epoch": int(e),
        "Training Loss": train_loss,
        "Validation Loss": eval_entry.get("eval_loss"),
        "Precision": eval_entry.get("eval_precision"),
        "Recall": eval_entry.get("eval_recall"),
        "F1": eval_entry.get("eval_f1"),
        "Accuracy": eval_entry.get("eval_accuracy"),
    }
    rows.append(row)

df_metrics = pd.DataFrame(rows).sort_values("Epoch")

df_metrics.head()

,Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
0,1,0.9305,0.645423,0.267857,0.361446,0.307692,0.815622
1,2,0.5092,0.416435,0.288732,0.493976,0.364444,0.882834
2,3,0.2850,0.361203,0.361538,0.566265,0.441315,0.896458
3,4,0.1871,0.361131,0.391667,0.566265,0.463054,0.892825
4,5,0.1317,0.363437,0.414414,0.554217,0.474227,0.899183


In [9]:
predictions, labels, _ = trainer.predict(tokenized_test_dataset)
predictions = np.argmax(predictions, axis=2)

y_true = [
    [id2label[l] for l in label if l != -100]
    for label in labels
]
y_pred = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]


In [8]:
def ids_to_tag_seqs(
    y_true: np.ndarray, 
    y_pred: np.ndarray, 
    id2label: dict
) -> Tuple[List[List[str]], List[List[str]]]:
    """
    Converte matrici [num_sentences, max_len] di ID in liste di liste di tag stringa,
    rimuovendo i token con label -100 (padding, subword, ecc.).
    """
    true_seqs = []
    pred_seqs = []

    for true_row, pred_row in zip(y_true, y_pred):
        seq_true = []
        seq_pred = []
        for t_id, p_id in zip(true_row, pred_row):
            if t_id == -100:   # ignora token
                continue
            seq_true.append(id2label[int(t_id)])
            seq_pred.append(id2label[int(p_id)])
        true_seqs.append(seq_true)
        pred_seqs.append(seq_pred)

    return true_seqs, pred_seqs


In [9]:
def plot_classification_report(
    y_pred: np.ndarray,
    y_true: np.ndarray,
    model_name: str,
    ax,
    id2label: dict
):
    """
    Classification report entity-level con seqeval, visualizzato come heatmap.
    """
    # 1) ID → BIO tag sequences
    true_seqs, pred_seqs = ids_to_tag_seqs(y_true, y_pred, id2label)

    # 2) classification_report di seqeval in forma dict
    #    output_dict=True → struttura simile a sklearn (per-classe + avg)
    cr_dict = classification_report(
        true_seqs,
        pred_seqs,
        output_dict=True,
        zero_division=0
    )

    # 3) DataFrame
    cr = pd.DataFrame(cr_dict).T

    # opzionale: tieni solo colonne numeriche principali
    keep_cols = [c for c in ["precision", "recall", "f1-score", "support"] if c in cr.columns]
    cr = cr[keep_cols].astype(float)

    # 4) Heatmap
    sns.heatmap(
        cr,
        annot=True,
        fmt=".2f",
        cmap="Greens",
        cbar=False,
        xticklabels=cr.columns,
        yticklabels=cr.index,
        linewidths=0.6,
        linecolor="black",
        ax=ax,
    )

    ax.set_facecolor("white")
    ax.set_title(f"CR (seqeval) {model_name}", fontsize=14, weight="bold")
    ax.tick_params(axis="x", labelsize=10, rotation=45)
    ax.tick_params(axis="y", labelsize=10)

In [10]:
plot_classification_report(y_pred, y_true, "NER_model", 0, )

TypeError: plot_classification_report() missing 1 required positional argument: 'id2label'

In [38]:
print(classification_report(y_true=y_true, y_pred=y_pred))

                precision    recall  f1-score   support

ACQUIRED_SKILL       0.53      0.67      0.59        15
   AVOID_SKILL       0.88      1.00      0.93         7
 INCLUDE_SKILL       0.48      0.54      0.51        57
 NEUTRAL_SKILL       0.42      0.57      0.48        14

     micro avg       0.51      0.60      0.55        93
     macro avg       0.58      0.70      0.63        93
  weighted avg       0.51      0.60      0.55        93



In [10]:
import itertools
from sklearn.metrics import confusion_matrix
import pandas as pd

# Flatten
y_true_flat = list(itertools.chain.from_iterable(y_true))
y_pred_flat = list(itertools.chain.from_iterable(y_pred))

def to_binary(label):
    # adattalo ai tuoi label (es. se hai "B-SKILL", "I-SKILL", ecc.)
    return "SKILL" if "SKILL" in label else "OTHER"

y_true_bin = [l for l in y_true_flat]
y_pred_bin = [l for l in y_pred_flat]

'''cm = confusion_matrix(y_true_bin, y_pred_bin, labels=["B-SKILL","I-SKILL", "O"])
df_cm = pd.DataFrame(cm, index=["B-SKILL_true", "I-SKILL_true", "O_true"],
                        columns=["B-SKILL_pred", "I-SKILL_pred", "O_pred"])
print(df_cm)'''

labels_cm = sorted(set(y_true_flat + y_pred_flat))
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=labels_cm)
df_cm = pd.DataFrame(cm, index=[f"{l}_true" for l in labels_cm],
                        columns=[f"{l}_pred" for l in labels_cm])

print(df_cm)


                        B-ACQUIRED_SKILL_pred  B-AMBIGUOUS_SKILL_pred  \
B-ACQUIRED_SKILL_true                      15                       0   
B-AMBIGUOUS_SKILL_true                      0                      11   
B-AVOID_SKILL_true                          0                       0   
B-INCLUDE_SKILL_true                        2                       2   
I-ACQUIRED_SKILL_true                       2                       0   
I-AMBIGUOUS_SKILL_true                      0                       0   
I-AVOID_SKILL_true                          0                       0   
I-INCLUDE_SKILL_true                        0                       0   
O_true                                     10                       0   

                        B-AVOID_SKILL_pred  B-INCLUDE_SKILL_pred  \
B-ACQUIRED_SKILL_true                    0                     1   
B-AMBIGUOUS_SKILL_true                   0                     1   
B-AVOID_SKILL_true                       9                     2 

In [11]:

error_examples = []
for i, (tl, tp) in enumerate(zip(y_true, y_pred)):
    if tl != tp:  
        error_examples.append((i, tl, tp))
        
for idx, tl, tp in error_examples[:5]:
    print(f"\nExample {idx}")
    print("TRUE: ", tl)
    print("PRED: ", tp)
    print(test_dataset[idx]["tokens"]) 



Example 2
TRUE:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'O', 'O', 'O', 'O', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'O', 'O', 'O', 'O', 'B-AMBIGUOUS_SKILL', 'I-AMBIGUOUS_SKILL', 'I-AMBIGUOUS_SKILL', 'O', 'O', 'O', 'O', 'O', 'O']
PRED:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'I-ACQUIRED_SKILL', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'O', 'B-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'I-INCLUDE_SKILL', 'O', 'O', 'O', 'O', 'B-AMBIGUOUS_SKILL', 'I-AMBIGUOUS_SKILL', 'I-AMBIGUOUS_SKILL', 'O', 'O', 'O', 'O', 'O', 'O']
['I', 'work', 'in', 'digital', 'services', 'and', 'Iâ€™ve', 'already', 'helped', 'rou